# Lab 4 - JAX and Ray: Qwen + LoRA at platform scale

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hyperscaleailabs/ml-platform-engineering/blob/main/notebooks/04_jax_ray_qwen_lora.ipynb)

**Runtime:** ~15 minutes on a laptop CPU (measured: Apple M4, default settings); a Colab T4
GPU is several times faster. Downloads ~1 GB, cached after the first run.
**Recommended:** Colab -> Runtime -> Change runtime type -> T4 GPU.

**Memory:** budget ~2 GB for the driver, ~2 GB for the shared copy of the weights in Ray's
object store, and ~2 GB of private JAX memory per worker - about 8 GB at the default of two
workers, which fits a standard Colab runtime. (Per-process RSS will *look* like 4 GB per
worker: the shared object-store mapping is counted in every process that maps it. That
double-counting is the whole point of the object store, and section 9 measures it.)
Set `LAB_SKIP_RAY=1` to skip that section entirely.

Lab 3 used HuggingFace end to end - a fine choice, and the right default. This lab rebuilds the
same workload on a different substrate, because the two answer different questions:

* **JAX** is a compiler-first framework. You write pure functions over pytrees; `jit` traces them
  into XLA graphs it can fuse, and `grad`/`vmap`/`shard_map` are transformations *of the function*
  rather than features of a class hierarchy. That model is what makes explicit device meshes and
  whole-program optimization tractable.
* **Ray** is the layer above a single training job: many jobs, many machines, shared state, and
  the scheduling and fault handling that goes with it.

## What you will do

1. Learn JAX's core transformations, including the traps that bite everyone once (`jit` on Python
   control flow, silent recompiles, functional PRNG).
2. **Reimplement Qwen2's forward pass in JAX** and verify it logit-for-logit against the
   HuggingFace PyTorch model. Every component is one you already wrote in Lab 2.
3. Shard a computation across a device mesh with `NamedSharding`.
4. Implement LoRA as a pytree, train it with `optax` under a jitted step function.
5. Evaluate with **first-token constrained scoring** - ~25,000x cheaper on the output projection
   than Lab 3's full-sequence scoring, and worth understanding exactly when it is valid.
6. Benchmark compile time against steady-state throughput.
7. Use **Ray** to run a LoRA hyperparameter sweep in parallel and a sharded distributed evaluation,
   with the base weights shared through the object store instead of copied per worker.

**Prerequisites:** Labs 2 and 3.

## 0. Setup

One environment variable must be set **before JAX is imported**: it exposes 8 virtual CPU devices
so the sharding section has a mesh to work with even on a single-GPU or CPU-only machine. If a GPU
is present JAX will still use it for real work - the virtual devices only affect the CPU backend.

In [ ]:
import os
import sys

os.environ.setdefault("XLA_FLAGS", "--xla_force_host_platform_device_count=8")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")   # required for Ray workers to share a GPU

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    os.system(f"{sys.executable} -m pip install -q "
              f"'transformers>=4.44' 'datasets>=2.20' optax 'ray>=2.9'")

import gc
import math
import time
import json
from functools import partial

import numpy as np
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
import optax
import torch
import transformers
import datasets as hfdatasets
from transformers import AutoTokenizer, AutoModelForCausalLM

# ---- Config (override via environment variables) -------------------------------------------------
MODEL_ID = os.environ.get("LAB_MODEL_ID", "Qwen/Qwen2.5-0.5B-Instruct")
SEED = int(os.environ.get("LAB_SEED", 0))
N_TRAIN = int(os.environ.get("LAB_N_TRAIN", 512))
N_EVAL = int(os.environ.get("LAB_N_EVAL", 128))
MAX_STEPS = int(os.environ.get("LAB_MAX_STEPS", 60))
BATCH_SIZE = int(os.environ.get("LAB_BATCH_SIZE", 4))
EVAL_BATCH = int(os.environ.get("LAB_EVAL_BATCH", 8))
SEQ_LEN = int(os.environ.get("LAB_SEQ_LEN", 112))     # fixed length: JAX recompiles per shape
LORA_R = int(os.environ.get("LAB_LORA_R", 16))
LORA_ALPHA = int(os.environ.get("LAB_LORA_ALPHA", 32))
LEARNING_RATE = float(os.environ.get("LAB_LR", 2e-4))
RAY_WORKERS = int(os.environ.get("LAB_RAY_WORKERS", 2))
RAY_TRIAL_STEPS = int(os.environ.get("LAB_RAY_TRIAL_STEPS", 30))
SKIP_RAY = os.environ.get("LAB_SKIP_RAY", "0") == "1"
# --------------------------------------------------------------------------------------------------

TF_MAJOR = int(transformers.__version__.split(".")[0])
DTYPE_KW = "dtype" if TF_MAJOR >= 5 else "torch_dtype"


def check(condition: bool, message: str) -> None:
    if not condition:
        raise AssertionError(f"CHECK FAILED: {message}")
    print(f"  ok - {message}")


print(f"jax          {jax.__version__}")
print(f"optax        {optax.__version__}")
print(f"transformers {transformers.__version__}")
print(f"backend      {jax.default_backend()}")
print(f"devices      {jax.devices()}")
print(f"cpu devices  {len(jax.devices('cpu'))} (virtual, for the sharding section)")
if jax.default_backend() == "cpu":
    print("\nNOTE: no accelerator found. Everything runs, but expect ~50 minutes. On Colab pick\n"
          "      Runtime -> Change runtime type -> T4 GPU, or lower LAB_MAX_STEPS / LAB_N_EVAL.")

## 1. JAX in ten minutes

JAX is NumPy plus four composable function transformations. The whole framework follows from one
constraint: **the functions you transform must be pure** - output depends only on the arguments, no
side effects, no mutation. That is what lets JAX trace a function once and hand the trace to a
compiler.

### Arrays are immutable

In [ ]:
x = jnp.arange(6.0).reshape(2, 3)
try:
    x[0, 0] = 99.0
except TypeError as exc:
    print(f"in-place assignment rejected: {exc}")

y = x.at[0, 0].set(99.0)     # functional update - returns a new array
print(f"\nx unchanged:\n{x}\ny updated:\n{y}")
check(x[0, 0] == 0.0 and y[0, 0] == 99.0, "the functional update leaves the original array alone")

### Randomness is explicit

There is no global RNG. You thread an explicit key, and **splitting is mandatory** - reusing a key
gives you the same numbers again. This looks like bureaucracy until you need a run to be
reproducible across 512 hosts, at which point it is the only design that works.

In [ ]:
key = jax.random.key(SEED)
a = jax.random.normal(key, (3,))
b = jax.random.normal(key, (3,))        # same key -> identical draw
k1, k2 = jax.random.split(key)
c = jax.random.normal(k1, (3,))

print(f"same key      : {a} vs {b}")
print(f"split key     : {c}")
check(jnp.allclose(a, b), "reusing a key reproduces the exact same values - a real bug source")
check(not jnp.allclose(a, c), "splitting produces independent randomness")

### `jit`, `grad`, `vmap`

In [ ]:
def predict(params, x):
    return jnp.tanh(x @ params["w"] + params["b"]).sum()


params = {"w": jax.random.normal(k1, (128, 128)), "b": jnp.zeros(128)}
xs = jax.random.normal(k2, (256, 128))

# grad differentiates with respect to argument 0, following any pytree structure.
grads = jax.grad(predict)(params, xs)
print(f"params is a pytree with leaves: {jax.tree.map(lambda t: t.shape, params)}")
print(f"grads mirrors it exactly:       {jax.tree.map(lambda t: t.shape, grads)}")

# vmap adds a batch dimension to a function written for a single example.
single = lambda v: jnp.tanh(v @ params["w"] + params["b"]).sum()
batched = jax.vmap(single)
print(f"\nvmap over 256 rows -> {batched(xs).shape}")
check(jnp.allclose(batched(xs)[0], single(xs[0]), atol=1e-5), "vmap matches the per-example function")

### What `jit` actually buys you

A common misconception is that `jit` makes the maths faster. It does not - your matmul already
calls the same tuned BLAS/cuBLAS kernel either way. What `jit` buys is **fusion**: eager mode
dispatches every elementwise op as its own kernel, each one reading its input from memory and
writing its output back. XLA fuses a chain of them into a single pass over the data.

So the payoff depends entirely on the shape of your computation. Measure both cases rather than
assuming.

In [ ]:
def timeit(fn, *args, repeats=50):
    jax.block_until_ready(fn(*args))          # warm up (and compile, if jitted)
    t0 = time.perf_counter()
    for _ in range(repeats):
        out = fn(*args)
    jax.block_until_ready(out)                # JAX dispatches asynchronously - never time without this
    return (time.perf_counter() - t0) / repeats


def matmul_bound(params, x):
    """One big matmul. Nothing to fuse - BLAS was already doing the work."""
    return jnp.tanh(x @ params["w"] + params["b"]).sum()


def fusion_bound(params, x):
    """A chain of cheap elementwise ops. Eager launches ~25 kernels; XLA fuses them into one."""
    h = x @ params["w"] + params["b"]
    for _ in range(6):
        h = jnp.tanh(h) * jax.nn.sigmoid(h) + 0.1 * jnp.sin(h)
    return (h ** 2).sum()


print(f"{'function':<16} {'compile':>10} {'eager':>11} {'jit':>11} {'speedup':>9} {'break-even':>12}")
timings = {}
for name, fn in [("matmul_bound", matmul_bound), ("fusion_bound", fusion_bound)]:
    compiled = jax.jit(fn)
    t0 = time.perf_counter()
    jax.block_until_ready(compiled(params, xs))
    compile_s = time.perf_counter() - t0

    t_eager, t_jit = timeit(fn, params, xs), timeit(compiled, params, xs)
    gain = t_eager / t_jit
    breakeven = compile_s / (t_eager - t_jit) if t_eager > t_jit else float("inf")
    timings[name] = (compile_s, t_eager, t_jit, gain)
    print(f"{name:<16} {compile_s * 1e3:9.1f}ms {t_eager * 1e6:10.1f}us {t_jit * 1e6:10.1f}us "
          f"{gain:8.2f}x {breakeven:>11,.0f} calls" if breakeven != float("inf")
          else f"{name:<16} {compile_s * 1e3:9.1f}ms {t_eager * 1e6:10.1f}us {t_jit * 1e6:10.1f}us "
               f"{gain:8.2f}x {'never':>17}")

check(timings["fusion_bound"][3] > 1.3,
      f"jit clearly wins on a fusable chain of elementwise ops "
      f"({timings['fusion_bound'][3]:.2f}x)")
check(timings["fusion_bound"][3] > timings["matmul_bound"][3],
      "...and wins less, or not at all, on a lone matmul - fusion is the mechanism")
check(timings["fusion_bound"][0] > timings["fusion_bound"][2],
      "compiling costs far more than one call, so jit only pays off inside a loop")

### The trap: `jit` traces on *shapes*, not values

Inside `jit`, array values are abstract tracers. Python `if` on a traced value fails, and every new
input **shape** triggers a fresh compilation. That is why this notebook pads to a fixed `SEQ_LEN`:
variable-length batches would recompile constantly and spend all their time in XLA.

In [ ]:
@jax.jit
def broken(v):
    if v.sum() > 0:            # Python branch on a traced value
        return v * 2
    return v


try:
    broken(jnp.ones(3))
except Exception as exc:
    print(f"{type(exc).__name__}: {str(exc).splitlines()[0]}")

fixed = jax.jit(lambda v: jnp.where(v.sum() > 0, v * 2, v))   # branch inside the graph
print(f"\njnp.where version works: {fixed(jnp.ones(3))}")

compile_count = 0
@jax.jit
def counted(v):
    global compile_count
    compile_count += 1          # runs only during tracing
    return v * 2


for shape in [(4,), (4,), (4,), (8,), (8,), (16,)]:
    counted(jnp.ones(shape))
print(f"\n6 calls across 3 distinct shapes -> {compile_count} compilations")
check(compile_count == 3, "one compilation per distinct input shape, cached thereafter")

## 2. Qwen2's forward pass in JAX

Now the real thing. Every function below is a component from Lab 2 - RMSNorm, RoPE, grouped-query
attention, SwiGLU - written functionally over a parameter pytree instead of as `nn.Module`s.

The success criterion is not "it runs". It is that the logits match HuggingFace's PyTorch
implementation to float32 precision. Without that check, a subtly wrong RoPE convention or a
transposed weight produces a model that looks plausible and is quietly broken.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

hf_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **{DTYPE_KW: torch.float32}).eval()
hf_cfg = hf_model.config

MODEL_CFG = {
    "n_layers": hf_cfg.num_hidden_layers,
    "n_heads": hf_cfg.num_attention_heads,
    "n_kv_heads": hf_cfg.num_key_value_heads,
    "d_head": hf_cfg.hidden_size // hf_cfg.num_attention_heads,
    "eps": hf_cfg.rms_norm_eps,
    "theta": float(getattr(hf_cfg, "rope_theta", None)
                   or (getattr(hf_cfg, "rope_parameters", None) or {}).get("rope_theta", 10_000.0)),
    "tied": bool(hf_cfg.tie_word_embeddings),
}
print(json.dumps(MODEL_CFG, indent=2))


def convert_weights(torch_model):
    """PyTorch state dict -> a JAX pytree.

    nn.Linear stores weight as (out, in) and computes x @ W.T; we transpose once at load time and
    use the (in, out) convention throughout, so every forward is a plain `x @ W`.
    """
    sd = torch_model.state_dict()
    take = lambda k: jnp.asarray(sd[k].detach().float().numpy())

    params = {"embed": take("model.embed_tokens.weight"),          # (vocab, hidden)
              "final_norm": take("model.norm.weight"),
              "layers": []}
    for i in range(MODEL_CFG["n_layers"]):
        p = f"model.layers.{i}."
        params["layers"].append({
            "attn_norm": take(p + "input_layernorm.weight"),
            # Qwen2 uses biases on q/k/v (Llama does not) - a classic porting mistake.
            "q_w": take(p + "self_attn.q_proj.weight").T, "q_b": take(p + "self_attn.q_proj.bias"),
            "k_w": take(p + "self_attn.k_proj.weight").T, "k_b": take(p + "self_attn.k_proj.bias"),
            "v_w": take(p + "self_attn.v_proj.weight").T, "v_b": take(p + "self_attn.v_proj.bias"),
            "o_w": take(p + "self_attn.o_proj.weight").T,
            "ffn_norm": take(p + "post_attention_layernorm.weight"),
            "gate_w": take(p + "mlp.gate_proj.weight").T,
            "up_w": take(p + "mlp.up_proj.weight").T,
            "down_w": take(p + "mlp.down_proj.weight").T,
        })
    # Tied embeddings: the LM head is the embedding table, so we never store a second copy.
    params["lm_head"] = None if MODEL_CFG["tied"] else take("lm_head.weight").T
    return params


t0 = time.time()
base_params = convert_weights(hf_model)
n_leaves = sum(leaf.size for leaf in jax.tree.leaves(base_params))
print(f"\nconverted {n_leaves / 1e6:.1f}M parameters in {time.time() - t0:.1f}s "
      f"({n_leaves * 4 / 1e9:.2f} GB as fp32)")

check(base_params["layers"][0]["q_w"].shape == (hf_cfg.hidden_size,
                                                MODEL_CFG["n_heads"] * MODEL_CFG["d_head"]),
      "the Q projection is stored (in, out) after transposition")
check(base_params["layers"][0]["k_w"].shape[1] == MODEL_CFG["n_kv_heads"] * MODEL_CFG["d_head"],
      "the K projection is sized by KV heads - GQA")

In [ ]:
def rms_norm(x, weight, eps):
    x32 = x.astype(jnp.float32)
    normed = x32 * jax.lax.rsqrt(jnp.mean(jnp.square(x32), axis=-1, keepdims=True) + eps)
    return normed * weight


def rotate_half(x):
    d = x.shape[-1] // 2
    return jnp.concatenate([-x[..., d:], x[..., :d]], axis=-1)


def rope_tables(seq_len: int, d_head: int, theta: float):
    inv_freq = 1.0 / (theta ** (jnp.arange(0, d_head, 2, dtype=jnp.float32) / d_head))
    freqs = jnp.outer(jnp.arange(seq_len, dtype=jnp.float32), inv_freq)
    emb = jnp.concatenate([freqs, freqs], axis=-1)          # HF "halves" convention, as in Lab 2
    return jnp.cos(emb), jnp.sin(emb)


def dense(x, w, b=None, lora=None, scaling=1.0):
    y = x @ w
    if b is not None:
        y = y + b
    if lora is not None:
        y = y + ((x @ lora["A"]) @ lora["B"]) * scaling      # (.., in) @ (in, r) @ (r, out)
    return y


def attention(h, p, lora, cos, sin, mask, cfg):
    B, T, _ = h.shape
    nh, nkv, hd = cfg["n_heads"], cfg["n_kv_heads"], cfg["d_head"]

    q = dense(h, p["q_w"], p["q_b"], lora and lora.get("q"), cfg["scaling"])
    k = dense(h, p["k_w"], p["k_b"], lora and lora.get("k"), cfg["scaling"])
    v = dense(h, p["v_w"], p["v_b"], lora and lora.get("v"), cfg["scaling"])

    q = q.reshape(B, T, nh, hd).transpose(0, 2, 1, 3)        # (B, Hq, T, hd)
    k = k.reshape(B, T, nkv, hd).transpose(0, 2, 1, 3)
    v = v.reshape(B, T, nkv, hd).transpose(0, 2, 1, 3)

    q = q * cos + rotate_half(q) * sin
    k = k * cos + rotate_half(k) * sin

    if nh != nkv:                                            # GQA: share each KV head across a group
        k = jnp.repeat(k, nh // nkv, axis=1)
        v = jnp.repeat(v, nh // nkv, axis=1)

    scores = (q @ k.transpose(0, 1, 3, 2)) / math.sqrt(hd)
    scores = jnp.where(mask, scores, jnp.finfo(scores.dtype).min)
    out = jax.nn.softmax(scores, axis=-1) @ v

    out = out.transpose(0, 2, 1, 3).reshape(B, T, nh * hd)
    return dense(out, p["o_w"], None, lora and lora.get("o"), cfg["scaling"])


def swiglu(h, p, lora, cfg):
    gate = dense(h, p["gate_w"], None, lora and lora.get("gate"), cfg["scaling"])
    up = dense(h, p["up_w"], None, lora and lora.get("up"), cfg["scaling"])
    return dense(jax.nn.silu(gate) * up, p["down_w"], None, lora and lora.get("down"), cfg["scaling"])


def forward_hidden(base, lora, input_ids, cfg):
    """Return the final hidden states (B, T, H). Note what this does NOT do: project to the
    151,936-way vocabulary. That projection is the single most expensive op in the model, and
    which positions you need it for depends on the task - so the caller decides."""
    B, T = input_ids.shape
    h = base["embed"][input_ids]

    cos, sin = rope_tables(T, cfg["d_head"], cfg["theta"])
    mask = jnp.tril(jnp.ones((T, T), dtype=bool))
    # Right padding only: with a causal mask, trailing pad tokens cannot influence earlier
    # positions, so no separate padding mask is needed. LEFT padding would require one.

    for i, layer in enumerate(base["layers"]):
        lp = lora["layers"][i] if lora is not None else None
        h = h + attention(rms_norm(h, layer["attn_norm"], cfg["eps"]), layer, lp, cos, sin, mask, cfg)
        h = h + swiglu(rms_norm(h, layer["ffn_norm"], cfg["eps"]), layer, lp, cfg)

    return rms_norm(h, base["final_norm"], cfg["eps"])


def project(base, hidden, token_subset=None):
    """Hidden states -> logits. `token_subset` restricts the output vocabulary, which turns a
    (896 x 151,936) matmul into (896 x 6) when you only need to compare a handful of candidates."""
    head = base["embed"] if base["lm_head"] is None else base["lm_head"].T   # (vocab, hidden)
    if token_subset is not None:
        head = head[token_subset]
    return hidden @ head.T


MODEL_CFG["scaling"] = LORA_ALPHA / LORA_R
jit_hidden = jax.jit(partial(forward_hidden, cfg=MODEL_CFG))

### The verification

Same tokens into both implementations, compare the logits. Float32 matmuls reassociate differently
across backends, so exact equality is not the bar - agreement to ~1e-3 on logits of magnitude ~20,
and identical argmax predictions, is.

In [ ]:
probe_text = "The capital of France is"
probe_ids = tokenizer(probe_text, return_tensors="np")["input_ids"]

with torch.no_grad():
    hf_logits = hf_model(torch.tensor(probe_ids)).logits.float().numpy()

jax_hidden = jit_hidden(base_params, None, jnp.asarray(probe_ids))
jax_logits = np.asarray(project(base_params, jax_hidden))

abs_diff = np.abs(hf_logits - jax_logits)
rel = abs_diff.max() / np.abs(hf_logits).max()
print(f"logits shape        {jax_logits.shape}")
print(f"max |difference|    {abs_diff.max():.4e}")
print(f"mean |difference|   {abs_diff.mean():.4e}")
print(f"logit magnitude     {np.abs(hf_logits).max():.2f}  -> relative error {rel:.2e}")
print(f"\nargmax agreement    {(hf_logits.argmax(-1) == jax_logits.argmax(-1)).mean():.1%}")
print(f"HuggingFace continues : {tokenizer.decode(hf_logits[0, -1].argmax())!r}")
print(f"JAX continues         : {tokenizer.decode(jax_logits[0, -1].argmax())!r}")

check(rel < 1e-3, "the JAX port matches HuggingFace to float32 precision")
check((hf_logits.argmax(-1) == jax_logits.argmax(-1)).all(), "every predicted token is identical")

top_hf = np.argsort(-hf_logits[0, -1])[:5]
top_jax = np.argsort(-jax_logits[0, -1])[:5]
print("\ntop-5 continuations:")
for r, (a, b) in enumerate(zip(top_hf, top_jax)):
    print(f"  {r + 1}. HF {tokenizer.decode(a)!r:<14} JAX {tokenizer.decode(b)!r}")
check(list(top_hf) == list(top_jax), "the full top-5 ranking agrees")

In [ ]:
# The PyTorch model has done its job as a reference. Free it before training.
del hf_model
gc.collect()

## 3. Sharding across a device mesh

`jit` is not limited to one device. You declare a **mesh** of devices with named axes, describe how
each array is laid out with `NamedSharding`, and XLA inserts the collectives. The same code runs on
1 device or 1024 - the sharding annotations change, the function does not.

Here we shard a batch across 8 virtual CPU devices along a `data` axis: replicate the parameters,
split the batch. That is plain data parallelism, and it is the layout most training jobs start with.

In [ ]:
from jax.sharding import Mesh, NamedSharding, PartitionSpec as P

cpu_devices = jax.devices("cpu")
mesh = Mesh(np.array(cpu_devices).reshape(-1), axis_names=("data",))
print(f"mesh over {len(cpu_devices)} devices: {mesh}")

batch = jax.device_put(jnp.arange(len(cpu_devices) * 4 * 16, dtype=jnp.float32)
                       .reshape(len(cpu_devices) * 4, 16),
                       NamedSharding(mesh, P("data", None)))     # split rows, replicate columns
weights = jax.device_put(jnp.ones((16, 16)), NamedSharding(mesh, P()))   # fully replicated

print(f"\nbatch  {batch.shape} sharded as {batch.sharding.spec}")
jax.debug.visualize_array_sharding(batch)
print(f"weights {weights.shape} replicated as {weights.sharding.spec}")


@jax.jit
def sharded_step(w, b):
    return jax.nn.relu(b @ w).sum(axis=-1)


out = sharded_step(weights, batch)
print(f"\noutput {out.shape} sharded as {out.sharding.spec}  <- XLA propagated the layout")

# Recompute the same thing on ONE device. Note that every array entering a single jit call must
# share a device layout: mixing a GPU-resident array with a CPU-sharded one is an error, and so is
# comparing an 8-device sharded result against a 1-device one with jnp.allclose. Pull both back to
# the host with np.asarray first - that gather is exactly what "bring the result home" means.
ref_w = jax.device_put(np.ones((16, 16), np.float32), cpu_devices[0])
ref_b = jax.device_put(np.asarray(batch), cpu_devices[0])
single_device = sharded_step(ref_w, ref_b)

check(np.allclose(np.asarray(out), np.asarray(single_device), atol=1e-4),
      "sharding changes where the work happens, not what the answer is")
check(len(batch.sharding.device_set) == len(cpu_devices), "the batch really is spread over every device")

> On a real cluster you extend the same idea: add a `model` axis for tensor parallelism
> (`P("data", "model")` on the weights), or shard the optimizer state alone for ZeRO-style memory
> savings. The mental model - name your axes, annotate your arrays, let the compiler place the
> collectives - does not change.

## 4. LoRA as a pytree

In PyTorch, LoRA means wrapping modules. In JAX there are no modules to wrap: the adapter is just a
second pytree, and `jax.grad` differentiates with respect to it while the base parameters ride
along as a constant argument. Freezing the base model requires no `requires_grad` flags and no
discipline - it is a consequence of which argument you differentiate.

In [ ]:
TARGETS = ["q", "k", "v", "o", "gate", "up", "down"]
SHAPE_OF = {"q": ("q_w",), "k": ("k_w",), "v": ("v_w",), "o": ("o_w",),
            "gate": ("gate_w",), "up": ("up_w",), "down": ("down_w",)}


def init_lora(key, base, r: int, targets=TARGETS):
    """A is drawn from a scaled normal, B starts at zero -> the adapter is a no-op at step 0."""
    layers = []
    for layer in base["layers"]:
        entry = {}
        for name in targets:
            w = layer[SHAPE_OF[name][0]]
            key, sub = jax.random.split(key)
            fan_in = w.shape[0]
            entry[name] = {
                "A": jax.random.normal(sub, (fan_in, r), dtype=jnp.float32) / math.sqrt(fan_in),
                "B": jnp.zeros((r, w.shape[1]), dtype=jnp.float32),
            }
        layers.append(entry)
    return {"layers": layers}


lora_params = init_lora(jax.random.key(SEED), base_params, LORA_R)
n_lora = sum(leaf.size for leaf in jax.tree.leaves(lora_params))
n_base = sum(leaf.size for leaf in jax.tree.leaves(base_params))
print(f"base   {n_base / 1e6:8.1f}M parameters")
print(f"LoRA   {n_lora / 1e6:8.3f}M parameters ({n_lora / n_base:.3%})")
print(f"adapter as fp32: {n_lora * 4 / 1e6:.1f} MB")

zero_adapter_hidden = jit_hidden(base_params, lora_params, jnp.asarray(probe_ids))
check(jnp.allclose(zero_adapter_hidden, jax_hidden, atol=1e-4),
      "B=0 means the adapter changes nothing before training, exactly as in Lab 3")

## 5. Data

Same task as Lab 3 - `dair-ai/emotion`, 6 classes - so the results are directly comparable.
Sequences are padded to a **fixed** `SEQ_LEN` because every distinct shape costs a recompilation.

In [ ]:
LABELS = ["sadness", "joy", "love", "anger", "fear", "surprise"]
SYSTEM_PROMPT = (
    "You are an emotion classifier. Classify the emotion expressed in the text as exactly one of: "
    + ", ".join(LABELS) + ". Reply with only that single word."
)

FALLBACK_ROWS = [
    ("i didnt feel humiliated", 0), ("i feel so blessed to have such good friends", 1),
    ("i am feeling romantic and tender toward him", 2), ("i feel furious that they lied to me", 3),
    ("i feel terrified about the exam tomorrow", 4), ("i feel shocked that it happened so fast", 5),
    ("i just feel really hopeless and lost", 0), ("i feel great about how the day went", 1),
    ("i feel such deep affection for my sister", 2), ("i am feeling irritated and angry at work", 3),
    ("i feel scared walking home in the dark", 4), ("i feel amazed by the sudden news", 5),
] * 60


def load_emotion():
    try:
        ds = hfdatasets.load_dataset("dair-ai/emotion", "split")
        return ds["train"], ds["test"]
    except Exception as exc:
        print(f"dataset download failed ({exc}); using the embedded fallback sample")
        rows = {"text": [t for t, _ in FALLBACK_ROWS], "label": [l for _, l in FALLBACK_ROWS]}
        full = hfdatasets.Dataset.from_dict(rows).shuffle(seed=SEED)
        split = full.train_test_split(test_size=0.3, seed=SEED)
        return split["train"], split["test"]


train_raw, test_raw = load_emotion()
train_ds = train_raw.shuffle(seed=SEED).select(range(min(N_TRAIN, len(train_raw))))
eval_ds = test_raw.shuffle(seed=SEED).select(range(min(N_EVAL, len(test_raw))))
print(f"train {len(train_ds)}   eval {len(eval_ds)}")


def build_prompt(text: str) -> str:
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": text}],
        tokenize=False, add_generation_prompt=True)


MAX_COMPLETION = max(len(tokenizer(l, add_special_tokens=False)["input_ids"]) for l in LABELS) + 1
print(f"longest label is {MAX_COMPLETION - 1} tokens; completions padded to {MAX_COMPLETION} (+EOS)")


def encode_example(text: str, label_idx: int):
    """Right-pad to SEQ_LEN and record exactly which positions carry loss.

    Rather than a full (SEQ_LEN, 151936) logit tensor we return the *indices* of the few positions
    that are supervised. Section 6 projects only those, which removes ~97% of the head's cost.
    """
    prompt_ids = tokenizer(build_prompt(text), add_special_tokens=False)["input_ids"]
    completion = tokenizer(LABELS[label_idx], add_special_tokens=False)["input_ids"] + [tokenizer.eos_token_id]

    ids = (prompt_ids + completion)[:SEQ_LEN]
    n_prompt = min(len(prompt_ids), SEQ_LEN)
    pad = SEQ_LEN - len(ids)
    input_ids = ids + [tokenizer.pad_token_id] * pad

    # Position n_prompt-1+j predicts completion token j.
    read_at, targets, mask = [], [], []
    for j in range(MAX_COMPLETION):
        pos = n_prompt - 1 + j
        valid = j < len(completion) and pos < SEQ_LEN - 1
        read_at.append(pos if valid else 0)
        targets.append(completion[j] if valid else 0)
        mask.append(1.0 if valid else 0.0)

    return (np.array(input_ids, np.int32), np.array(read_at, np.int32),
            np.array(targets, np.int32), np.array(mask, np.float32), np.int32(n_prompt - 1))


def encode_split(ds):
    cols = list(zip(*[encode_example(t, l) for t, l in zip(ds["text"], ds["label"])]))
    keys = ["input_ids", "read_at", "targets", "mask", "prompt_end"]
    return {k: np.stack(v) for k, v in zip(keys, cols)}


train_arrays = encode_split(train_ds)
eval_arrays = encode_split(eval_ds)
eval_gold = np.array(eval_ds["label"])

print(f"input_ids {train_arrays['input_ids'].shape}  supervised positions per example "
      f"{train_arrays['mask'].sum(1).mean():.1f} of {SEQ_LEN}")

# Truncation is silent by construction: an example whose prompt overflows SEQ_LEN ends up fully
# masked and contributes nothing to the loss, while still occupying a slot in every batch. Measure
# it. A padded-out dataset that trains on 60% of its examples is a bug that looks like a bad model.
for split, arrays in (("train", train_arrays), ("eval", eval_arrays)):
    supervised = (arrays["mask"].sum(1) > 0)
    print(f"{split:>6}: {supervised.mean():.1%} of examples carry loss, "
          f"prompt end at position {arrays['prompt_end'].max()} of {SEQ_LEN} at worst")

check(train_arrays["mask"].sum() > 0, "some positions are supervised")
check(train_arrays["input_ids"].shape[1] == SEQ_LEN, "every sequence is exactly SEQ_LEN long")
check((train_arrays["mask"].sum(1) > 0).mean() > 0.95,
      "at least 95% of training examples fit in SEQ_LEN and carry loss - raise LAB_SEQ_LEN if not")

## 6. Training

The whole training step - forward, loss, gradients, optimizer update - compiles into **one** XLA
executable. `donate_argnums` tells XLA that the old parameters and optimizer state will not be used
again, so it may reuse their buffers instead of allocating new ones. At scale that donation is the
difference between fitting in memory and not.

In [ ]:
def loss_fn(lora, base, batch, cfg):
    hidden = forward_hidden(base, lora, batch["input_ids"], cfg)
    # Gather only the supervised positions, then project just those. (B, MAX_COMPLETION, H)
    picked = jnp.take_along_axis(hidden, batch["read_at"][:, :, None], axis=1)
    logits = project(base, picked)
    per_token = optax.softmax_cross_entropy_with_integer_labels(logits, batch["targets"])
    return (per_token * batch["mask"]).sum() / jnp.maximum(batch["mask"].sum(), 1.0)


def make_train_step(cfg, optimizer):
    @partial(jax.jit, donate_argnums=(0, 1))
    def step(lora, opt_state, base, batch):
        loss, grads = jax.value_and_grad(loss_fn)(lora, base, batch, cfg)
        updates, opt_state = optimizer.update(grads, opt_state, lora)
        return optax.apply_updates(lora, updates), opt_state, loss
    return step


schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0, peak_value=LEARNING_RATE,
    warmup_steps=max(1, MAX_STEPS // 20), decay_steps=MAX_STEPS, end_value=LEARNING_RATE * 0.1)
optimizer = optax.chain(optax.clip_by_global_norm(1.0), optax.adamw(schedule, weight_decay=0.0))

opt_state = optimizer.init(lora_params)
train_step = make_train_step(MODEL_CFG, optimizer)


def batches(arrays, batch_size, steps, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(arrays["input_ids"])
    for _ in range(steps):
        idx = rng.integers(0, n, size=batch_size)
        yield {k: jnp.asarray(v[idx]) for k, v in arrays.items()}


print(f"training {MAX_STEPS} steps, batch {BATCH_SIZE}, seq {SEQ_LEN}\n")
losses, t_start, first_step_s = [], time.time(), None
for i, batch in enumerate(batches(train_arrays, BATCH_SIZE, MAX_STEPS)):
    t_step = time.time()
    lora_params, opt_state, loss = train_step(lora_params, opt_state, base_params, batch)
    loss = float(loss)
    if i == 0:
        first_step_s = time.time() - t_step      # includes tracing + XLA compilation
    losses.append(loss)
    if i % max(1, MAX_STEPS // 10) == 0 or i == MAX_STEPS - 1:
        print(f"step {i:4d}  loss {loss:.4f}  lr {float(schedule(i)):.2e}  "
              f"{time.time() - t_start:6.1f}s")

steady_s = (time.time() - t_start - first_step_s) / max(1, MAX_STEPS - 1)
print(f"\nfirst step {first_step_s:.1f}s (compile) vs steady state {steady_s:.2f}s/step "
      f"-> compilation cost {first_step_s / steady_s:.0f} steps")

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(losses, lw=1)
ax.plot(np.convolve(losses, np.ones(5) / 5, mode="valid"), lw=2, label="5-step mean")
ax.set_xlabel("step"); ax.set_ylabel("masked cross-entropy"); ax.legend()
ax.set_title("LoRA training in JAX"); plt.tight_layout(); plt.show()

check(np.isfinite(losses).all(), "no NaNs - the loss stayed finite")
check(np.mean(losses[-5:]) < np.mean(losses[:5]), "the loss decreased")

## 7. Evaluation: first-token constrained scoring

Lab 3 scored every label by summing log-probabilities over all of its tokens - 6 forward passes per
example. Here we exploit a property of *this* label set: all six labels begin with a **distinct**
first token. So one forward pass gives all six scores, read off the last prompt position.

Then we go further. We do not need 151,936 logits to compare 6 candidates - just the 6 rows of the
output projection that matter. The head matmul shrinks from `896 x 151,936` to `896 x 6`.

**When is this valid?** Only when the first tokens are distinct, which the check below enforces
rather than assumes. When they are not (label sets sharing a prefix, or multi-word answers), you
must fall back to full-sequence scoring. Getting this wrong silently degrades every number you
report, so verify it, never eyeball it.

In [ ]:
first_token_ids = np.array([tokenizer(l, add_special_tokens=False)["input_ids"][0] for l in LABELS])
print(f"{'label':<10} {'first token':>12}  decoded")
for name, tid in zip(LABELS, first_token_ids):
    print(f"{name:<10} {tid:>12}  {tokenizer.decode(int(tid))!r}")

check(len(set(first_token_ids.tolist())) == len(LABELS),
      "all six labels start with a DISTINCT token - first-token scoring is valid here")

label_tokens = jnp.asarray(first_token_ids)


def make_scorer(cfg):
    @jax.jit
    def score(base, lora, input_ids, prompt_end):
        hidden = forward_hidden(base, lora, input_ids, cfg)
        last = jnp.take_along_axis(hidden, prompt_end[:, None, None], axis=1)[:, 0]   # (B, H)
        return project(base, last, token_subset=label_tokens)                          # (B, 6)
    return score


scorer = make_scorer(MODEL_CFG)


def evaluate(base, lora, arrays, gold, batch_size=EVAL_BATCH):
    preds = []
    for start in range(0, len(gold), batch_size):
        sl = slice(start, start + batch_size)
        ids = jnp.asarray(arrays["input_ids"][sl])
        ends = jnp.asarray(arrays["prompt_end"][sl])
        if ids.shape[0] < batch_size:        # pad the final batch to keep one compiled shape
            pad = batch_size - ids.shape[0]
            ids = jnp.concatenate([ids, jnp.zeros((pad, SEQ_LEN), ids.dtype)])
            ends = jnp.concatenate([ends, jnp.zeros((pad,), ends.dtype)])
        scores = scorer(base, lora, ids, ends)
        preds.extend(np.asarray(scores).argmax(-1)[:min(batch_size, len(gold) - start)])
    preds = np.array(preds)
    return float((preds == gold).mean()), preds


t0 = time.time()
acc_base, preds_base = evaluate(base_params, None, eval_arrays, eval_gold)
t_base_eval = time.time() - t0

t0 = time.time()
acc_lora, preds_lora = evaluate(base_params, lora_params, eval_arrays, eval_gold)
t_lora_eval = time.time() - t0

majority = float(np.bincount(eval_gold, minlength=6).max() / len(eval_gold))
print(f"\n{'setting':<28} {'accuracy':>10}")
print(f"{'majority class':<28} {majority:>9.2%}")
print(f"{'base (zero-shot)':<28} {acc_base:>9.2%}   ({t_base_eval:.1f}s)")
print(f"{'LoRA fine-tuned':<28} {acc_lora:>9.2%}   ({t_lora_eval:.1f}s)")

check(acc_lora > acc_base, "LoRA training improved accuracy over the base model")
check(acc_lora > majority, "the fine-tuned model beats the majority-class baseline")

In [ ]:
# Confirm the cheap head is exactly as accurate as the expensive one - the optimization must be
# free, not approximately free.
sample = slice(0, min(16, len(eval_gold)))
ids_s = jnp.asarray(eval_arrays["input_ids"][sample])
ends_s = jnp.asarray(eval_arrays["prompt_end"][sample])

hidden_s = jit_hidden(base_params, lora_params, ids_s)
last_s = jnp.take_along_axis(hidden_s, ends_s[:, None, None], axis=1)[:, 0]
full_logits = project(base_params, last_s)                       # all 151,936
subset_logits = project(base_params, last_s, token_subset=label_tokens)

full_choice = np.asarray(full_logits)[:, first_token_ids].argmax(-1)
subset_choice = np.asarray(subset_logits).argmax(-1)
print(f"full-vocabulary head : {full_logits.shape}")
print(f"6-token head         : {subset_logits.shape}  "
      f"({full_logits.size / subset_logits.size:,.0f}x fewer output values)")
check(np.array_equal(full_choice, subset_choice),
      "restricting the output projection changes nothing about the predictions")

## 8. Benchmarking JAX

Two numbers matter and they pull in opposite directions: **compile time** (paid per distinct input
shape) and **steady-state throughput**. A service that sees many shapes can spend more time
compiling than computing, which is why production JAX serving buckets sequence lengths into a small
fixed set - each bucket compiles once and is then reused forever.

In [ ]:
def bench(fn, *args, warmup=2, repeats=5):
    for _ in range(warmup):
        jax.block_until_ready(fn(*args))
    t0 = time.perf_counter()
    for _ in range(repeats):
        out = fn(*args)
    jax.block_until_ready(out)
    return (time.perf_counter() - t0) / repeats


print(f"{'batch':>6} {'compile (s)':>12} {'run (s)':>9} {'seq/s':>8} {'tokens/s':>10}")
rows = []
for bs in [1, 2, 4, 8]:
    ids = jnp.asarray(eval_arrays["input_ids"][:1]).repeat(bs, axis=0)
    ends = jnp.zeros((bs,), jnp.int32) + int(eval_arrays["prompt_end"][0])
    fn = make_scorer(MODEL_CFG)        # a fresh jit cache, so we measure compilation honestly
    t0 = time.perf_counter()
    jax.block_until_ready(fn(base_params, lora_params, ids, ends))
    compile_s = time.perf_counter() - t0
    run_s = bench(fn, base_params, lora_params, ids, ends)
    rows.append((bs, compile_s, run_s, bs / run_s, bs * SEQ_LEN / run_s))
    print(f"{bs:>6} {compile_s:>12.2f} {run_s:>9.4f} {bs / run_s:>8.1f} {bs * SEQ_LEN / run_s:>10.0f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot([r[0] for r in rows], [r[4] for r in rows], "o-")
axes[0].set_xlabel("batch size"); axes[0].set_ylabel("tokens/s"); axes[0].set_title("Throughput")
axes[1].plot([r[0] for r in rows], [r[1] for r in rows], "o-", label="compile")
axes[1].plot([r[0] for r in rows], [r[2] for r in rows], "o-", label="steady-state run")
axes[1].set_yscale("log"); axes[1].set_xlabel("batch size"); axes[1].set_ylabel("seconds")
axes[1].set_title("Compile vs run"); axes[1].legend()
plt.tight_layout(); plt.show()

print(f"\nbatching {rows[-1][0]}x improved throughput {rows[-1][4] / rows[0][4]:.1f}x")
print(f"compiling one shape costs {rows[-1][1] / rows[-1][2]:.0f} steady-state calls")
check(rows[-1][4] > rows[0][4], "larger batches deliver more tokens per second")
check(rows[0][1] > rows[0][2], "compilation is far more expensive than a single call")

## 9. Ray: many jobs, not one

Everything so far occupied a single process. Ray is the layer that turns it into a fleet:

* `@ray.remote` functions are **tasks** - stateless units the scheduler places on any worker.
* `ray.put` places an object in the shared **object store** once; workers on the same node map it
  in rather than each deserializing a copy. For a 2 GB parameter set across N workers, that is the
  difference between `2 GB` and `N x 2 GB` of RAM.
* `runtime_env` controls worker environment variables - here `XLA_PYTHON_CLIENT_PREALLOCATE=false`,
  without which the first JAX worker would grab most of the GPU and the second would fail to start.

We run the rank ablation from Lab 3's exercises as parallel trials.

> **Memory note.** Each worker materializes its own JAX copy of the base weights (~2 GB fp32), so
> `LAB_RAY_WORKERS` defaults to 2. Raise it if you have RAM to spare, or set `LAB_SKIP_RAY=1` to
> skip this section entirely.

In [ ]:
if SKIP_RAY:
    print("LAB_SKIP_RAY=1 - skipping the Ray section.")
else:
    import ray

    ray.init(ignore_reinit_error=True, num_cpus=RAY_WORKERS, include_dashboard=False,
             log_to_driver=False,
             runtime_env={"env_vars": {"XLA_PYTHON_CLIENT_PREALLOCATE": "false"}})
    print(json.dumps(ray.cluster_resources(), indent=2))

In [ ]:
if not SKIP_RAY:
    # Ship the weights as NumPy once. Workers rebuild the JAX pytree from the shared buffers.
    base_numpy = jax.tree.map(lambda a: np.asarray(a), base_params)
    payload_mb = sum(a.nbytes for a in jax.tree.leaves(base_numpy)) / 1e6

    weights_ref = ray.put(base_numpy)
    train_ref = ray.put(train_arrays)
    eval_ref = ray.put(eval_arrays)
    gold_ref = ray.put(eval_gold)

    # ray.put copies into the object store, so the driver now holds the weights three times over:
    # the JAX arrays, this NumPy view, and the store's copy. Drop the one we no longer need.
    # Forgetting this is a routine way to double a driver's memory for no benefit.
    del base_numpy
    gc.collect()

    print(f"placed {payload_mb:.0f} MB of weights in the object store, shared by every worker")

    @ray.remote
    def lora_trial(weights, train, evals, gold, hp, cfg, seq_len, label_ids, steps, batch_size):
        """One (rank, learning rate) trial: train an adapter, return its accuracy.

        Runs in a separate process, so it re-imports JAX and rebuilds the pytree from the
        object-store buffers. Everything it needs is passed explicitly - the classic
        stateless-task shape that lets the scheduler retry it anywhere.
        """
        import jax, jax.numpy as jnp, numpy as np, optax, math, time
        from functools import partial

        cfg = dict(cfg)
        cfg["scaling"] = hp["alpha"] / hp["r"]
        base = jax.tree.map(jnp.asarray, weights)

        key = jax.random.key(hp["seed"])
        lora = init_lora(key, base, hp["r"])

        sched = optax.warmup_cosine_decay_schedule(0.0, hp["lr"], max(1, steps // 20), steps,
                                                   end_value=hp["lr"] * 0.1)
        opt = optax.chain(optax.clip_by_global_norm(1.0), optax.adamw(sched, weight_decay=0.0))
        state = opt.init(lora)

        @partial(jax.jit, donate_argnums=(0, 1))
        def step(lora, state, base, batch):
            loss, grads = jax.value_and_grad(loss_fn)(lora, base, batch, cfg)
            updates, state = opt.update(grads, state, lora)
            return optax.apply_updates(lora, updates), state, loss

        rng = np.random.default_rng(hp["seed"])
        n = len(train["input_ids"])
        t0 = time.time()
        losses = []
        for _ in range(steps):
            idx = rng.integers(0, n, size=batch_size)
            batch = {k: jnp.asarray(v[idx]) for k, v in train.items()}
            lora, state, loss = step(lora, state, base, batch)
            losses.append(float(loss))

        @jax.jit
        def score(base, lora, ids, ends):
            from_hidden = forward_hidden(base, lora, ids, cfg)
            last = jnp.take_along_axis(from_hidden, ends[:, None, None], axis=1)[:, 0]
            head = base["embed"] if base["lm_head"] is None else base["lm_head"].T
            return last @ head[jnp.asarray(label_ids)].T

        preds = []
        bs = 8
        for start in range(0, len(gold), bs):
            ids = jnp.asarray(evals["input_ids"][start:start + bs])
            ends = jnp.asarray(evals["prompt_end"][start:start + bs])
            if ids.shape[0] < bs:
                pad = bs - ids.shape[0]
                ids = jnp.concatenate([ids, jnp.zeros((pad, seq_len), ids.dtype)])
                ends = jnp.concatenate([ends, jnp.zeros((pad,), ends.dtype)])
            preds.extend(np.asarray(score(base, lora, ids, ends)).argmax(-1)[:min(bs, len(gold) - start)])

        n_params = sum(leaf.size for leaf in jax.tree.leaves(lora))
        return {**hp, "accuracy": float((np.array(preds) == gold).mean()),
                "final_loss": float(np.mean(losses[-5:])), "params": n_params,
                "seconds": time.time() - t0}

In [ ]:
if not SKIP_RAY:
    SWEEP = [{"r": 4, "alpha": 8, "lr": 2e-4, "seed": SEED},
             {"r": 16, "alpha": 32, "lr": 2e-4, "seed": SEED},
             {"r": 4, "alpha": 8, "lr": 1e-3, "seed": SEED},
             {"r": 16, "alpha": 32, "lr": 1e-3, "seed": SEED}]

    cfg_plain = {k: v for k, v in MODEL_CFG.items() if k != "scaling"}
    t0 = time.time()
    futures = [lora_trial.remote(weights_ref, train_ref, eval_ref, gold_ref, hp, cfg_plain,
                                 SEQ_LEN, first_token_ids, RAY_TRIAL_STEPS, BATCH_SIZE)
               for hp in SWEEP]
    results = ray.get(futures)          # blocks until all trials finish
    wall = time.time() - t0

    results.sort(key=lambda r: -r["accuracy"])
    print(f"{'rank':>5} {'alpha':>6} {'lr':>8} {'params':>9} {'loss':>7} {'accuracy':>9} {'trial s':>8}")
    for r in results:
        print(f"{r['r']:>5} {r['alpha']:>6} {r['lr']:>8.0e} {r['params']:>9,} "
              f"{r['final_loss']:>7.3f} {r['accuracy']:>8.2%} {r['seconds']:>8.1f}")

    serial = sum(r["seconds"] for r in results)
    print(f"\nwall clock {wall:.1f}s for {len(SWEEP)} trials; summed trial time {serial:.1f}s")
    print(f"speedup {serial / wall:.2f}x on {RAY_WORKERS} workers "
          f"(ideal {min(RAY_WORKERS, len(SWEEP))}x)")
    print("\nThe gap comes from three places, and it is worth knowing which one you are paying:")
    print("  1. worker startup - a fresh process importing JAX, once per worker;")
    print("  2. XLA compilation - each worker compiles the step function for itself, and the")
    print("     compilation cache is per process, so N workers pay it N times;")
    print("  3. resource contention - on a CPU-only machine every worker's XLA thread pool wants")
    print("     all the cores, so two 'parallel' trials mostly take turns. On a real cluster with")
    print("     one accelerator per worker this term disappears; on one laptop it dominates.")
    print("\nThis is why sweeps are worth distributing and single short jobs usually are not:")
    print("fixed per-worker costs are amortized only when each trial is long enough to matter.")

    check(len(results) == len(SWEEP), "every trial returned a result")
    check(all(np.isfinite(r["accuracy"]) for r in results), "every trial produced a finite accuracy")

    fig, ax = plt.subplots(figsize=(6.5, 3.8))
    for lr in sorted({r["lr"] for r in results}):
        pts = sorted([r for r in results if r["lr"] == lr], key=lambda r: r["r"])
        ax.plot([p["r"] for p in pts], [p["accuracy"] for p in pts], "o-", label=f"lr={lr:.0e}")
    ax.set_xlabel("LoRA rank"); ax.set_ylabel("accuracy"); ax.set_xscale("log", base=2)
    ax.set_title(f"Ray sweep - {len(SWEEP)} trials on {RAY_WORKERS} workers"); ax.legend()
    plt.tight_layout(); plt.show()

### Distributed evaluation

The sweep parallelizes over *configurations*. The other axis is parallelizing over *data*: shard
the evaluation set, score each shard on a different worker, concatenate. This is the pattern behind
every large offline scoring job, and its correctness requirement is easy to state and easy to
violate - the sharded result must be **identical** to the single-process result, so we check it.

In [ ]:
if not SKIP_RAY:
    @ray.remote
    def eval_shard(weights, evals, gold, shard_idx, n_shards, cfg, seq_len, label_ids):
        import jax, jax.numpy as jnp, numpy as np
        base = jax.tree.map(jnp.asarray, weights)
        cfg = dict(cfg); cfg["scaling"] = 1.0

        lo = shard_idx * len(gold) // n_shards
        hi = (shard_idx + 1) * len(gold) // n_shards

        @jax.jit
        def score(base, ids, ends):
            hidden = forward_hidden(base, None, ids, cfg)
            last = jnp.take_along_axis(hidden, ends[:, None, None], axis=1)[:, 0]
            head = base["embed"] if base["lm_head"] is None else base["lm_head"].T
            return last @ head[jnp.asarray(label_ids)].T

        preds, bs = [], 8
        for start in range(lo, hi, bs):
            stop = min(start + bs, hi)
            ids = jnp.asarray(evals["input_ids"][start:stop])
            ends = jnp.asarray(evals["prompt_end"][start:stop])
            take = stop - start
            if take < bs:
                ids = jnp.concatenate([ids, jnp.zeros((bs - take, seq_len), ids.dtype)])
                ends = jnp.concatenate([ends, jnp.zeros((bs - take,), ends.dtype)])
            preds.extend(np.asarray(score(base, ids, ends)).argmax(-1)[:take])
        return shard_idx, np.array(preds)

    t0 = time.time()
    shard_futures = [eval_shard.remote(weights_ref, eval_ref, gold_ref, i, RAY_WORKERS,
                                       cfg_plain, SEQ_LEN, first_token_ids)
                     for i in range(RAY_WORKERS)]
    shards = sorted(ray.get(shard_futures), key=lambda s: s[0])
    distributed_preds = np.concatenate([s[1] for s in shards])
    dist_time = time.time() - t0

    print(f"distributed over {RAY_WORKERS} shards in {dist_time:.1f}s "
          f"(single process took {t_base_eval:.1f}s)")
    print(f"accuracy {float((distributed_preds == eval_gold).mean()):.2%} "
          f"vs single-process {acc_base:.2%}")

    check(np.array_equal(distributed_preds, preds_base),
          "the sharded evaluation reproduces the single-process predictions exactly")

    ray.shutdown()
    print("\nray shut down")

## 10. Exercises

1. **Break the port.** Change `rotate_half` to the interleaved RoPE convention
   (`x[..., ::2]` / `x[..., 1::2]`). The logit check in section 2 will fail - by how much, and does
   the model still produce fluent text? This is what a silently wrong port looks like.
2. **Drop the q/k/v biases.** Qwen2 has them, Llama does not. Set them to zero and measure the
   logit error. Would you have noticed without the reference check?
3. **bfloat16.** Cast `base_params` to `jnp.bfloat16` and re-run the verification. Quantify the
   logit error and the speedup, and decide whether the trade is acceptable for this task.
4. **Tensor parallelism.** Extend the mesh in section 3 with a `model` axis and shard `gate_w` and
   `up_w` column-wise, `down_w` row-wise. Verify the outputs are unchanged.
5. **Recompilation audit.** Set `JAX_LOG_COMPILES=1` and run section 7 with a ragged final batch
   (remove the padding-to-batch-size logic). Count the extra compilations.
6. **KV cache in JAX.** Add incremental decoding with a preallocated cache updated via
   `jax.lax.dynamic_update_slice` - a fixed-shape buffer, because a growing one would recompile
   every step. Compare against Lab 2's PyTorch cache.
7. **Ray Tune.** Replace the fixed grid in section 9 with `ray.tune` and an ASHA scheduler so bad
   trials are stopped early. Compare total compute against the exhaustive grid.
8. **Fault tolerance.** Make `lora_trial` raise on one configuration and observe how `ray.get`
   surfaces it. Then add `max_retries` and a try/except so one bad trial cannot sink the sweep.

## What you built

* A Qwen2 forward pass in JAX, verified logit-for-logit against HuggingFace - the only standard
  that catches a wrong RoPE convention or a transposed weight.
* LoRA as a pytree, where freezing the base model is a consequence of what you differentiate
  rather than a flag you must remember to set.
* A single fused training step under `jit` with donated buffers.
* An evaluation that projects 6 output rows instead of 151,936, with a check that the shortcut is
  exact and a statement of when it stops being valid.
* Sharding across a device mesh, and the compile-time-versus-throughput trade that governs JAX in
  production.
* Ray tasks sharing 2 GB of weights through the object store, running a parallel hyperparameter
  sweep and a distributed evaluation verified against the single-process result.

## The series, end to end

| Lab | Built | Verified against |
|---|---|---|
| 1 | Autograd, training loop, MLP | Hand-derived gradients |
| 2 | Transformer: attention, RoPE, GQA, RMSNorm, SwiGLU, KV cache | PyTorch reference kernels |
| 3 | LoRA fine-tune of Qwen2.5-0.5B | PEFT, and a few-shot baseline |
| 4 | Qwen2 in JAX, LoRA, sharding, Ray | HuggingFace logits, single-process eval |

The thread running through all four: **build the thing, then prove it is right against something
independent.** In a notebook that is a `check(...)`. In production it is a regression test, a
golden-output comparison, or a shadow deployment - but it is the same discipline, and it is what
separates an ML platform that can be changed safely from one that cannot.